# Breakout DQN — train & evaluate

This notebook walks through the full pipeline: building the Atari environment, instantiating an agent (DQN or Double DQN), training it for a small number of episodes, and evaluating a trained checkpoint.

For longer runs, use the CLI scripts in `scripts/` instead — they accept the same hyperparameters as flags.

## 1. Imports & path setup

In [ ]:
import os, sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))

import torch
import numpy as np
import matplotlib.pyplot as plt

from utils import make_env, show_observation_stack
from dqn_cnn_model import DQN_CNN_Model
from dqn_agent import DQNAgent
from double_dqn_agent import DoubleDQNAgent

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print('device:', device)

## 2. Build the environment

`make_env` applies the standard Atari DQN preprocessing: frame skipping, grayscale + resize to 84×84, frame stacking (4 frames), and reward clipping.

In [ ]:
env = make_env('ALE/Breakout-v5')
print('observation shape:', env.observation_space.shape)
print('actions:', env.action_space.n)

obs, _ = env.reset()
show_observation_stack(obs)

## 3. Train a Double DQN agent (short demo run)

Real training takes many thousands of episodes. This cell trains for only 50 episodes so the notebook stays runnable — use `scripts/train.py` for the real thing.

In [ ]:
def state_to_tensor(s):
    return torch.tensor(s, dtype=torch.float32) / 255.0

obs_shape = env.observation_space.shape
n_actions = env.action_space.n

online = DQN_CNN_Model(obs_shape, n_actions)
target = DQN_CNN_Model(obs_shape, n_actions)
target.load_state_dict(online.state_dict())

agent = DoubleDQNAgent(
    env, online, target, state_to_tensor,
    memory_buffer_size=10_000, batch_size=32,
    learning_rate=1e-4, gamma=0.99,
    epsilon_i=1.0, epsilon_f=0.1, epsilon_anneal_steps=100_000,
    episode_block=10, device=device, sync_target=500,
)
agent.checkpoint_dir = '../checkpoints'

rewards, avg_q = agent.train(number_episodes=50, max_steps=20_000)

## 4. Plot training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(rewards); ax1.set_xlabel('episode'); ax1.set_title('Episode reward')
ax2.plot(avg_q); ax2.set_xlabel('episode'); ax2.set_title('Average Q')
plt.tight_layout(); plt.show()

## 5. Evaluate a trained checkpoint

Load one of the pre-trained checkpoints shipped with this repo and play a few episodes greedily.

In [ ]:
eval_env = make_env('ALE/Breakout-v5')
model = DQN_CNN_Model(eval_env.observation_space.shape, eval_env.action_space.n).to(device)
model.load_state_dict(torch.load('../checkpoints/double_dqn_final.dat', map_location=device))
model.eval()

scores = []
for ep in range(3):
    s, _ = eval_env.reset()
    done, total = False, 0.0
    while not done:
        x = state_to_tensor(s).unsqueeze(0).to(device)
        with torch.no_grad():
            a = int(model(x).argmax(dim=1).item())
        s, r, term, trunc, _ = eval_env.step(a)
        done = term or trunc
        total += r
    scores.append(total)
    print(f'episode {ep+1}: reward = {total}')

print(f'mean: {np.mean(scores):.2f}')